# Optimal-decay matrix control barrier functions

Companion notebook to *Optimal Decay Matrix Control Barrier Functions: Soundness and
Completeness for Matrix Safety Constraints* (S. G. Gessow, A. Wu, A. D. Ames and B. T. Lopez,
[arXiv:2610.18675](https://arxiv.org/abs/2610.18675)).

Start Jupyter from the repository root (`jupyter lab`) and run the cells in order. The notebook

1. sets up the singularity-avoidance constraint for a planar two-link arm,
2. compares the nominal, standard MCBF, scalar OD, and matrix OD controllers in closed loop,
3. animates the four closed loops, and
4. draws the timelapse and closed-loop figures.

The arm follows `two_link_manipulator_kinematics.md`, in absolute joint angles, as in the paper.
This closed loop, with a constant command, is not in the paper;
`scripts/reproduce_paper.py` reproduces the paper's Section VIII.

Requirements: see the README for the virtual environment. MOSEK is used when installed (it
needs a license) and CLARABEL otherwise.

In [ ]:
import warnings

import numpy as np
import cvxpy as cp
from IPython.display import HTML

from odmcbf import SafetyFilter, TwoLinkArm, run_comparison, summarize
from odmcbf import figures

# Setup of the closed-loop comparison.
GAMMA   = 1.0                       # alpha(r) = gamma r
OMEGA_D = 1.0                       # nominal decay rate
RHO     = 5e-3                      # rho_omega; rho_Omega = (4/p) rho_omega = 1e-2
U_NOM   = np.array([1.0, 0.4])      # commanded joint rates (theta1dot, theta2dot)  [rad/s]
THETA0  = np.array([0.0, np.pi/2])  # absolute joint angles (theta1, theta2)  [rad]
DT, HORIZON = 0.01, 8.0

arm = TwoLinkArm(gamma=GAMMA)       # unit links; sigma_bar puts the boundary at |theta2 - theta1| = pi/3
print(arm)
# CLARABEL reports a few steps as optimal_inaccurate; the filter accepts them and
# checks the barrier LMI itself (FilterResult.safety_residual), so the warning is muted.
warnings.filterwarnings("ignore", message="Solution may be inaccurate", category=UserWarning)
print("solver:", "MOSEK" if "MOSEK" in cp.installed_solvers() else "CLARABEL",
      "| cvxpy", cp.__version__, "| numpy", np.__version__)

## 1. The constraint

The arm has unit links, absolute joint angles $\theta=(\theta_1,\theta_2)$, both measured
counter-clockwise from the horizontal, and velocity control $\dot\theta=\mathbf u$. With
Jacobian $J(\theta)$ and $M=JJ^\top$, the matrix constraint

$$H(\theta)=M(\theta)-\bar\sigma^2 I\succeq 0
\iff \sigma_{\min}(J(\theta))\ge\bar\sigma$$

keeps the arm away from singular configurations. The eigenvalues of $H$ depend on $\theta$
only through $\theta_2-\theta_1$. The threshold $\bar\sigma^2=1/2$ puts the boundary of the
safe set at $|\theta_2-\theta_1|=\pi/3$ and $2\pi/3$.

Advancing both joints together rotates the whole arm about joint 1 and conjugates $H$ by a
rotation: **that motion never changes the safety margin.**

In [ ]:
lo, hi = arm.safe_theta_diff_interval
print(f"sigma_bar^2 = {arm.sigma_bar_sq:.6f};  safe set {lo:.4f} <= |theta2 - theta1| <= {hi:.4f}")
for theta1 in (0.0, 0.55, 1.30):                 # the spectrum is independent of theta1
    print(f"  eig H(theta1={theta1:+.2f}, theta2 - theta1 = {lo:.4f}) = "
          f"{np.linalg.eigvalsh(arm.H([theta1, theta1 + lo]))}")

step = 1e-6
slope = (arm.margin([0.4, 0.4 + lo + step]) - arm.margin([0.4, 0.4 + lo - step])) / (2*step)
lam = arm.l1**2 + arm.l2**2 - 2*arm.sigma_bar_sq          # nonzero eigenvalue of H on the boundary
print(f"\nd lambda_min(H) / d(theta2 - theta1) at the boundary = {slope:.6f}"
      f"  (= l1^2 l2^2 sin(2*{lo:.4f}) / lambda = {arm.l1**2 * arm.l2**2 * np.sin(2*lo)/lam:.6f})")
closest = np.full(2, U_NOM.mean())
print("so the tangency condition at the boundary is theta2dot - theta1dot >= 0, with the rotation")
print(f"of the whole arm unrestricted: the control closest to ({U_NOM[0]:g}, {U_NOM[1]:g}) "
      f"satisfying it is ({closest[0]:g}, {closest[1]:g})")

## 2. Closed-loop comparison

Starting from $\theta(0)=(0,\pi/2)$, where the margin is largest, the commanded joint rates
$\mathbf k_{\rm nom}=(1,0.4)$ rad/s are applied for 8 s. Joint 1 turns at 1 rad/s while
$\theta_2-\theta_1$ closes at 0.6 rad/s, so the arm runs into the boundary
$\theta_2-\theta_1=\pi/3$ before reaching the stretched singular configuration
$\theta_2=\theta_1$.

A minimally invasive filter should stop $\theta_2-\theta_1$ from closing and keep rotating
the whole arm about joint 1; the closest such command is $(0.7,0.7)$ rad/s.

In [ ]:
traces = run_comparison(arm, u_nom=U_NOM, theta0=THETA0, dt=DT, horizon=HORIZON,
                        omega_d=OMEGA_D, rho_omega=RHO, recovery=True)
summary = {name: summarize(trace, U_NOM, OMEGA_D) for name, trace in traces.items()}

header = (f"{'controller':<16}{'min margin':>12}{'joint-1 turn [rad]':>20}"
          f"{'late u':>18}{'decay dev':>11}{'solve [ms]':>12}")
print(header); print("-"*len(header))
for name, trace in traces.items():
    s = summary[name]
    fmt = lambda x, w, p=2: f"{x:>{w}.{p}f}" if x is not None else f"{'-':>{w}}"
    late_u = trace.u[-round(2.0/DT):].mean(axis=0)
    late = f"({late_u[0]:.3f}, {late_u[1]:.3f})"
    print(f"{trace.label:<16}{s['min_margin']:>12.2e}{s['joint1_rotation']:>20.2f}"
          f"{late:>18}{fmt(s['decay_deviation_settled'],11)}{fmt(s['mean_solve_ms'],12)}")

t_hit = (THETA0[1] - THETA0[0] - lo) / (U_NOM[0] - U_NOM[1])   # nominal reaches the boundary
best = U_NOM[0]*t_hit + U_NOM.mean()*(HORIZON - t_hit)
print(f"\ncommanded: {U_NOM[0]*HORIZON:.2f} rad of joint-1 rotation; switching to the closest safe "
      f"command at the boundary gives {best:.2f} rad.")
print("per-step solve times are machine dependent")

## 3. Animation

Each panel shows the arm, the singular-value ellipse at the end-effector, and the dashed
threshold circle. The constraint is active exactly when the ellipse's minor axis touches
the circle.

In [ ]:
HTML(figures.animate(traces, arm=arm, skip=8).to_jshtml(fps=12, default_mode="loop"))

## 4. Figures

In [ ]:
figures.snapshots(traces, arm=arm)     # timelapse of the four closed loops

In [ ]:
figures.closed_loop(traces, U_NOM, omega_d=OMEGA_D)     # margin, control and decay deviations